# Phase 2: GPU Naive Implementation
**CSC14120 - Parallel Programming**

## Hướng dẫn:
1. Zip thư mục project (không bao gồm data/)
2. Upload file zip lên Colab
3. Chạy tất cả cells

In [ ]:
# Kiểm tra GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Upload va giai nen project
from google.colab import files
import zipfile, os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

project_root = None
for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        project_root = root
        break

if project_root is None:
    raise RuntimeError("Could not find project root containing 'src' directory")

os.chdir(project_root)
print("Current working directory:", os.getcwd())
!ls

In [ ]:
# Download CIFAR-10
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Done!')
!ls data/*.bin

In [ ]:
# Build Phase 2 (GPU Naive)
!nvcc -O2 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr -Iinclude \
    -o gpu_train src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/dataset.cpp
print('Build complete!')

In [ ]:
# Train
!./gpu_train --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase2.csv --log-txt phase2.txt --save-weights phase2.weights

In [ ]:
# Visualize
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase2.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'b-o'); ax1.set_title('Loss'); ax1.grid(True)
ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'g-o'); ax2.set_title('Time (s)'); ax2.grid(True)
plt.tight_layout(); plt.show()

print(f"Best Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Avg Time: {ep['epoch_time_sec'].mean():.2f}s/epoch")
print(f"Total: {ep['epoch_time_sec'].sum():.2f}s")

In [ ]:
# Download results
files.download('phase2.csv')
files.download('phase2.txt')
files.download('phase2.weights')